# SageMaker Explorer — 조회 & 엔드포인트 호출

이 노트북은 AWS SageMaker 리소스를 조회하고, vLLM 엔드포인트를 직접 호출하는 전체 흐름을 다룹니다.

**사전 조건:**
- IAM 유저 또는 EC2 인스턴스 롤에 아래 권한 필요:
  - `sagemaker:ListEndpoints`
  - `sagemaker:DescribeEndpoint`
  - `sagemaker:ListModels`
  - `sagemaker:DescribeModel`
  - `sagemaker:InvokeEndpoint`
- `~/.aws/credentials` 또는 EC2 Instance Role 설정

## Cell 1. Config — 여기만 수정하세요

In [9]:
# ============================
#  설정값
# ============================
REGION        = "ap-northeast-2"          # AWS 리전
ENDPOINT_NAME = "dots-mocr-vllm-endpoint" # 조회 및 호출할 SageMaker 엔드포인트 이름

# vLLM 호출 파라미터
MODEL_NAME  = "model"   # vllm serve --served-model-name 에 설정한 값
MAX_TOKENS  = 512       # 테스트용 짧게 설정
TEMPERATURE = 0.1

# 테스트 이미지 (None 이면 단순 텍스트 요청으로 대체)
TEST_IMAGE_PATH = None  # ex) "./sample.png"
TEST_PROMPT     = "이 이미지의 내용을 간단히 설명해 주세요."

print(f"Region  : {REGION}")
print(f"Endpoint: {ENDPOINT_NAME}")

Region  : ap-northeast-2
Endpoint: dots-mocr-vllm-endpoint


## Cell 2. boto3 클라이언트 초기화

In [10]:
import boto3
import json
import base64
import io
from IPython.display import display, Markdown
from pprint import pprint

# ──────────────────────────────────────────────────────────────
# EC2 인스턴스 롤 사용 (credential_source = Ec2InstanceMetadata)
#
# ~/.aws/credentials 의 IAM 유저 키(default 프로파일)는
# SageMaker 권한이 없고, EC2 롤보다 우선순위가 높아서 덮어씁니다.
# → profile_name='ec2_role' 을 명시해 EC2 롤 임시 자격증명을 사용합니다.
#
# ~/.aws/config 에 아래 내용이 등록되어 있어야 합니다:
#   [profile ec2_role]
#   credential_source = Ec2InstanceMetadata
#   region = ap-northeast-2
# ──────────────────────────────────────────────────────────────
session = boto3.Session(profile_name="ec2_role", region_name=REGION)

# 관리용 클라이언트 (리소스 조회)
sm    = session.client("sagemaker")

# 런타임 클라이언트 (엔드포인트 호출)
sm_rt = session.client("sagemaker-runtime")

# 현재 자격증명 확인
sts = session.client("sts")
try:
    identity = sts.get_caller_identity()
    print("✅ AWS 자격증명 확인 (EC2 인스턴스 롤)")
    print(f"  Account : {identity['Account']}")
    print(f"  ARN     : {identity['Arn']}")
except Exception as e:
    print(f"❌ 자격증명 오류: {e}")
    print("  → ~/.aws/config 에 [profile ec2_role] 섹션이 있는지 확인하세요.")

✅ AWS 자격증명 확인 (EC2 인스턴스 롤)
  Account : 847776736936
  ARN     : arn:aws:sts::847776736936:assumed-role/seokhyeon_ec2_iamRole/i-0c64de1663fcc6477


## Cell 3. 전체 엔드포인트 목록 조회

In [11]:
try:
    paginator = sm.get_paginator("list_endpoints")
    all_endpoints = []
    for page in paginator.paginate(SortBy="CreationTime", SortOrder="Descending"):
        all_endpoints.extend(page["Endpoints"])

    if not all_endpoints:
        print("⚠️  엔드포인트가 없습니다.")
    else:
        print(f"총 {len(all_endpoints)}개 엔드포인트:\n")
        print(f"{'이름':<45} {'상태':<15} {'생성일'}")
        print("-" * 85)
        for ep in all_endpoints:
            status = ep["EndpointStatus"]
            icon   = "🟢" if status == "InService" else ("🔴" if status == "Failed" else "🟡")
            print(f"{ep['EndpointName']:<45} {icon} {status:<12} {ep['CreationTime'].strftime('%Y-%m-%d %H:%M')}")
except Exception as e:
    print(f"❌ 엔드포인트 목록 조회 실패: {e}")

총 1개 엔드포인트:

이름                                            상태              생성일
-------------------------------------------------------------------------------------
dots-mocr-vllm-endpoint                       🟢 InService    2026-04-06 02:42


## Cell 4. 특정 엔드포인트 상세 조회

In [12]:
try:
    ep_detail = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)

    status = ep_detail["EndpointStatus"]
    icon   = "🟢" if status == "InService" else ("🔴" if status == "Failed" else "🟡")

    print(f"{icon} 엔드포인트: {ep_detail['EndpointName']}")
    print(f"  상태          : {status}")
    print(f"  ARN           : {ep_detail['EndpointArn']}")
    print(f"  엔드포인트Config: {ep_detail['EndpointConfigName']}")
    print(f"  생성일         : {ep_detail['CreationTime']}")
    print(f"  최종수정       : {ep_detail['LastModifiedTime']}")

    if "ProductionVariants" in ep_detail:
        print("\n프로덕션 배리언트:")
        for v in ep_detail["ProductionVariants"]:
            print(f"  - {v['VariantName']}")
            print(f"      인스턴스타입 : {v.get('CurrentInstanceType', 'N/A')}")
            print(f"      인스턴스수   : {v.get('CurrentInstanceCount', 'N/A')}")
            print(f"      가중치       : {v.get('CurrentWeight', 'N/A')}")

    IS_IN_SERVICE = (status == "InService")
    if not IS_IN_SERVICE:
        print(f"\n⚠️  엔드포인트가 InService 상태가 아닙니다. 호출하려면 InService 상태가 필요합니다.")

except sm.exceptions.ClientError as e:
    if "Could not find endpoint" in str(e):
        print(f"❌ 엔드포인트 '{ENDPOINT_NAME}'을 찾을 수 없습니다.")
        print("  ENDPOINT_NAME 값을 Cell 1에서 수정하세요.")
        IS_IN_SERVICE = False
    else:
        print(f"❌ 오류: {e}")
        IS_IN_SERVICE = False
except Exception as e:
    print(f"❌ 오류: {e}")
    IS_IN_SERVICE = False

🟢 엔드포인트: dots-mocr-vllm-endpoint
  상태          : InService
  ARN           : arn:aws:sagemaker:ap-northeast-2:847776736936:endpoint/dots-mocr-vllm-endpoint
  엔드포인트Config: dots-mocr-vllm-endpoint
  생성일         : 2026-04-06 02:42:22.682000+00:00
  최종수정       : 2026-04-06 02:50:12.331000+00:00

프로덕션 배리언트:
  - AllTraffic
      인스턴스타입 : N/A
      인스턴스수   : 1
      가중치       : 1.0


## Cell 5. 엔드포인트 Config 조회

In [13]:
try:
    ep_detail = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    config_name = ep_detail["EndpointConfigName"]

    config = sm.describe_endpoint_config(EndpointConfigName=config_name)
    print(f"엔드포인트 Config: {config_name}")
    print(f"  ARN: {config['EndpointConfigArn']}")
    print(f"  KMS Key: {config.get('KmsKeyId', '없음')}")

    print("\n프로덕션 배리언트 설정:")
    for v in config["ProductionVariants"]:
        print(f"  배리언트명    : {v['VariantName']}")
        print(f"  모델명        : {v['ModelName']}")
        print(f"  인스턴스타입  : {v.get('InstanceType', 'N/A')}")
        print(f"  초기인스턴스수: {v.get('InitialInstanceCount', 'N/A')}")
        print(f"  초기가중치    : {v.get('InitialVariantWeight', 'N/A')}")
        print()

        # 해당 모델 정보도 조회
        try:
            model_info = sm.describe_model(ModelName=v["ModelName"])
            print(f"  모델 ARN      : {model_info['ModelArn']}")
            pc = model_info.get("PrimaryContainer", {})
            print(f"  컨테이너 이미지: {pc.get('Image', 'N/A')}")
            env = pc.get("Environment", {})
            if env:
                print(f"  환경변수:")
                for k, val in env.items():
                    print(f"    {k} = {val}")
        except Exception as me:
            print(f"  모델 조회 오류: {me}")

except Exception as e:
    print(f"❌ Config 조회 실패: {e}")

엔드포인트 Config: dots-mocr-vllm-endpoint
  ARN: arn:aws:sagemaker:ap-northeast-2:847776736936:endpoint-config/dots-mocr-vllm-endpoint
  KMS Key: 없음

프로덕션 배리언트 설정:
  배리언트명    : AllTraffic
  모델명        : vllm-2026-04-06-02-42-21-503
  인스턴스타입  : ml.g6.2xlarge
  초기인스턴스수: 1
  초기가중치    : 1.0

  모델 ARN      : arn:aws:sagemaker:ap-northeast-2:847776736936:model/vllm-2026-04-06-02-42-21-503
  컨테이너 이미지: 763104351884.dkr.ecr.ap-northeast-2.amazonaws.com/vllm:0.17.1-gpu-py312-cu129-ubuntu22.04-sagemaker
  환경변수:
    SM_VLLM_CHAT_TEMPLATE_CONTENT_FORMAT = string
    SM_VLLM_GPU_MEMORY_UTILIZATION = 0.9
    SM_VLLM_MAX_MODEL_LEN = 8192
    SM_VLLM_MODEL = rednote-hilab/dots.mocr
    SM_VLLM_SERVED_MODEL_NAME = model
    SM_VLLM_TRUST_REMOTE_CODE =  


## Cell 6. 전체 모델 목록 조회

In [14]:
try:
    paginator = sm.get_paginator("list_models")
    all_models = []
    for page in paginator.paginate(SortBy="CreationTime", SortOrder="Descending"):
        all_models.extend(page["Models"])

    if not all_models:
        print("⚠️  등록된 모델이 없습니다.")
    else:
        print(f"총 {len(all_models)}개 모델:\n")
        print(f"{'모델명':<55} {'생성일'}")
        print("-" * 80)
        for m in all_models[:20]:  # 최대 20개 표시
            print(f"{m['ModelName']:<55} {m['CreationTime'].strftime('%Y-%m-%d %H:%M')}")
        if len(all_models) > 20:
            print(f"  ... 외 {len(all_models)-20}개")
except Exception as e:
    print(f"❌ 모델 목록 조회 실패: {e}")

총 1개 모델:

모델명                                                     생성일
--------------------------------------------------------------------------------
vllm-2026-04-06-02-42-21-503                            2026-04-06 02:42


## Cell 7. 엔드포인트 호출 — 텍스트 요청 (이미지 없이)

In [15]:
import json

# vLLM OpenAI 호환 포맷 — 텍스트 전용
text_payload = {
    "model": MODEL_NAME,
    "messages": [
        {
            "role": "user",
            "content": "안녕하세요! 간단한 테스트입니다. '동작 확인'이라고 답해 주세요."
        }
    ],
    "max_tokens": MAX_TOKENS,
    "temperature": TEMPERATURE,
}

print(f"엔드포인트 '{ENDPOINT_NAME}'에 텍스트 요청 전송 중...")
print(f"페이로드: {json.dumps(text_payload, ensure_ascii=False, indent=2)}\n")

try:
    response = sm_rt.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps(text_payload),
    )
    raw = response["Body"].read().decode("utf-8")
    result = json.loads(raw)

    print("✅ 응답 수신")
    print(f"\n--- 원본 응답 ---")
    print(json.dumps(result, ensure_ascii=False, indent=2))

    # OpenAI 호환 응답에서 텍스트 추출
    if isinstance(result, dict) and "choices" in result:
        content = result["choices"][0]["message"]["content"]
        print(f"\n--- 모델 응답 텍스트 ---")
        print(content)
    else:
        print(f"\n--- 응답 (파싱 불가, 원문) ---")
        print(raw[:1000])

except Exception as e:
    print(f"❌ 호출 실패: {e}")
    print("\n👉 확인사항:")
    print("  1. ENDPOINT_NAME 이 올바른지 확인")
    print("  2. 엔드포인트가 InService 상태인지 확인 (Cell 4 참고)")
    print("  3. IAM 권한에 sagemaker:InvokeEndpoint 포함 여부 확인")

엔드포인트 'dots-mocr-vllm-endpoint'에 텍스트 요청 전송 중...
페이로드: {
  "model": "model",
  "messages": [
    {
      "role": "user",
      "content": "안녕하세요! 간단한 테스트입니다. '동작 확인'이라고 답해 주세요."
    }
  ],
  "max_tokens": 512,
  "temperature": 0.1
}

✅ 응답 수신

--- 원본 응답 ---
{
  "id": "chatcmpl-81a4690d72b80206",
  "object": "chat.completion",
  "created": 1776130346,
  "model": "model",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "안녕! '동작 확인'이라고 답해 주세요.",
        "refusal": null,
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [],
        "reasoning": null
      },
      "logprobs": null,
      "finish_reason": "stop",
      "stop_reason": 151673,
      "token_ids": null
    }
  ],
  "service_tier": null,
  "system_fingerprint": null,
  "usage": {
    "prompt_tokens": 27,
    "total_tokens": 42,
    "completion_tokens": 15,
    "prompt_tokens_details": null
  },
  "prompt_logprobs": null

## Cell 8. 엔드포인트 호출 — 이미지 포함 멀티모달 요청

In [ ]:
import base64
import io
from pathlib import Path

# ──────────────────────────────────────────────────
# 이미지 준비: TEST_IMAGE_PATH 가 없으면 더미 이미지 생성
# ──────────────────────────────────────────────────
try:
    from PIL import Image, ImageDraw, ImageFont
    HAS_PIL = True
except ImportError:
    HAS_PIL = False
    print("⚠️  Pillow 미설치. 텍스트 전용 요청만 가능합니다.")

if HAS_PIL:
    if TEST_IMAGE_PATH and Path(TEST_IMAGE_PATH).exists():
        img = Image.open(TEST_IMAGE_PATH).convert("RGB")
        print(f"이미지 로드: {TEST_IMAGE_PATH}  ({img.size})")
    else:
        # 테스트용 더미 이미지 (흰 배경 + 텍스트)
        img = Image.new("RGB", (400, 200), color=(255, 255, 255))
        draw = ImageDraw.Draw(img)
        draw.text((20, 80), "SageMaker Endpoint Test", fill=(0, 0, 0))
        draw.text((20, 110), "안녕하세요 테스트 이미지입니다.", fill=(50, 50, 200))
        print("더미 테스트 이미지 생성 (400x200)")

    # base64 인코딩
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    print(f"이미지 base64 크기: {len(img_b64):,} 글자")

    display(img)

    # ──────────────────────────────────────────────────
    # vLLM 멀티모달 페이로드 구성 (OpenAI 호환)
    # ──────────────────────────────────────────────────
    vision_payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{img_b64}"},
                    },
                    {
                        "type": "text",
                        "text": TEST_PROMPT,
                    },
                ],
            }
        ],
        "max_tokens": MAX_TOKENS,
        "temperature": TEMPERATURE,
    }

    print(f"\n엔드포인트 '{ENDPOINT_NAME}'에 이미지+텍스트 요청 전송 중...")
    print(f"프롬프트: {TEST_PROMPT}\n")

    try:
        response = sm_rt.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps(vision_payload),
        )
        raw = response["Body"].read().decode("utf-8")
        result = json.loads(raw)

        print("✅ 응답 수신")

        if isinstance(result, dict) and "choices" in result:
            content = result["choices"][0]["message"]["content"]
            usage   = result.get("usage", {})
            print(f"\n--- 모델 응답 ---")
            print(content)
            if usage:
                print(f"\n토큰 사용량: 입력={usage.get('prompt_tokens','?')}  출력={usage.get('completion_tokens','?')}")
        else:
            print(f"\n--- 응답 원문 ---")
            print(raw[:2000])

    except Exception as e:
        print(f"❌ 호출 실패: {e}")

## Cell 9. dots.ocr 전용 — 레이아웃 OCR 호출 (prompt_layout_all_en)

In [ ]:
# dots.ocr 엔드포인트에 특화된 레이아웃 OCR 호출
# PROMPT_MODE 에 따라 다른 프롬프트를 전송합니다

DOTS_OCR_PROMPT = """Detect the layout of the given document image and extract all text. \
Return a JSON array where each element is an object with keys: \
category (string), bbox ([x1,y1,x2,y2] normalized 0-1), text (string)."""

if HAS_PIL:
    # 이미지 base64 재사용 (Cell 8 에서 생성)
    layout_payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{img_b64}"},
                    },
                    {
                        "type": "text",
                        "text": f"<|img|><|imgpad|><|endofimg|>{DOTS_OCR_PROMPT}",
                    },
                ],
            }
        ],
        "max_tokens": 4096,
        "temperature": 0.1,
    }

    print(f"dots.ocr 레이아웃 OCR 요청 전송 중...\n")

    try:
        response = sm_rt.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps(layout_payload),
        )
        raw = response["Body"].read().decode("utf-8")
        result = json.loads(raw)

        print("✅ 응답 수신")

        if isinstance(result, dict) and "choices" in result:
            content = result["choices"][0]["message"]["content"]
            print(f"\n--- 레이아웃 JSON 응답 ---")
            print(content[:2000])

            # JSON 파싱 시도
            import re, ast
            json_match = re.search(r"```(?:json)?\s*([\s\S]+?)```", content)
            json_str = json_match.group(1).strip() if json_match else content.strip()
            try:
                cells = json.loads(json_str)
                print(f"\n파싱된 레이아웃 블록 수: {len(cells)}")
                for i, cell in enumerate(cells[:5]):
                    print(f"  [{i+1}] category={cell.get('category','?')}  "
                          f"bbox={cell.get('bbox','?')}  "
                          f"text={str(cell.get('text',''))[:60]}")
                if len(cells) > 5:
                    print(f"  ... 외 {len(cells)-5}개")
            except Exception:
                print("JSON 파싱 실패 — 원문 확인 필요")
        else:
            print(raw[:2000])

    except Exception as e:
        print(f"❌ 호출 실패: {e}")
else:
    print("Pillow 미설치로 이미지 호출 불가")

## Cell 10. 엔드포인트 모니터링 — CloudWatch 메트릭 조회

In [ ]:
import datetime

cw = session.client("cloudwatch")

METRICS = [
    ("Invocations",         "Sum"),
    ("ModelLatency",        "Average"),
    ("OverheadLatency",     "Average"),
    ("Invocation4XXErrors", "Sum"),
    ("Invocation5XXErrors", "Sum"),
]

end_time   = datetime.datetime.utcnow()
start_time = end_time - datetime.timedelta(hours=24)  # 최근 24시간

print(f"기간: {start_time.strftime('%Y-%m-%d %H:%M')} ~ {end_time.strftime('%Y-%m-%d %H:%M')} UTC\n")

for metric_name, stat in METRICS:
    try:
        resp = cw.get_metric_statistics(
            Namespace="AWS/SageMaker",
            MetricName=metric_name,
            Dimensions=[
                {"Name": "EndpointName", "Value": ENDPOINT_NAME},
                {"Name": "VariantName",  "Value": "AllTraffic"},
            ],
            StartTime=start_time,
            EndTime=end_time,
            Period=3600,  # 1시간 단위
            Statistics=[stat],
        )
        datapoints = sorted(resp["Datapoints"], key=lambda x: x["Timestamp"])
        if datapoints:
            total  = sum(dp[stat] for dp in datapoints)
            latest = datapoints[-1][stat]
            unit   = datapoints[-1].get("Unit", "")
            print(f"  {metric_name:<25} 합계={total:.1f}  최근값={latest:.1f} {unit}")
        else:
            print(f"  {metric_name:<25} 데이터 없음")
    except Exception as e:
        print(f"  {metric_name:<25} 조회 실패: {e}")

## Cell 11. 권한 진단

In [ ]:
import datetime

# 현재 자격증명으로 어떤 SageMaker 작업이 가능한지 확인
checks = [
    ("sagemaker:ListEndpoints",        lambda: sm.list_endpoints(MaxResults=1)),
    ("sagemaker:ListModels",           lambda: sm.list_models(MaxResults=1)),
    ("sagemaker:DescribeEndpoint",     lambda: sm.describe_endpoint(EndpointName=ENDPOINT_NAME)),
    ("sagemaker:InvokeEndpoint",       lambda: sm_rt.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"model": MODEL_NAME, "messages": [{"role": "user", "content": "ping"}], "max_tokens": 1})
    )),
    ("cloudwatch:GetMetricStatistics", lambda: session.client("cloudwatch").get_metric_statistics(
        Namespace="AWS/SageMaker", MetricName="Invocations",
        Dimensions=[{"Name": "EndpointName", "Value": ENDPOINT_NAME}, {"Name": "VariantName", "Value": "AllTraffic"}],
        StartTime=datetime.datetime.utcnow() - datetime.timedelta(hours=1),
        EndTime=datetime.datetime.utcnow(), Period=3600, Statistics=["Sum"]
    )),
]

print(f"{'권한 체크':<40} {'결과'}")
print("-" * 60)
for action, fn in checks:
    try:
        fn()
        print(f"  {action:<38} ✅ OK")
    except Exception as e:
        err = str(e)
        if "AccessDenied" in err or "not authorized" in err:
            print(f"  {action:<38} ❌ 권한 없음")
        elif "Could not find" in err or "ValidationException" in err:
            print(f"  {action:<38} ⚠️  리소스 없음 (권한은 OK)")
        else:
            print(f"  {action:<38} ⚠️  {err[:60]}")

print()
print("필요한 IAM 정책 (최소 권한):")
policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Action": [
        "sagemaker:ListEndpoints",
        "sagemaker:ListModels",
        "sagemaker:DescribeEndpoint",
        "sagemaker:DescribeEndpointConfig",
        "sagemaker:DescribeModel",
        "sagemaker:InvokeEndpoint",
    ], "Resource": "*"}]
}
print(json.dumps(policy, indent=2, ensure_ascii=False))